In [0]:
%sql
-- Cria o schema (dataset)
CREATE SCHEMA IF NOT EXISTS workspace.nyc_taxi

In [0]:
%sql
-- Cria os volumes de armazenamento dos dados brutos
CREATE VOLUME IF NOT EXISTS workspace.nyc_taxi.landing_zone;

In [0]:
# Os arquivos foram inseridos na landing zone manualmente
# Verifica os arquivos na landing zone
display(dbutils.fs.ls("/Volumes/workspace/nyc_taxi/landing_zone/"))

In [0]:
from pyspark.sql import functions as f
from pyspark.sql import types as t

# Lê os arquivos parquet da landing zone e define o schema explicitamente das colunas necessárias para que não haja inconsistências nos tipos dos campos

LANDING_PATH = "/Volumes/workspace/nyc_taxi/landing_zone/"

df_nyc_taxi_data_final = None

for month in ["01", "02", "03", "04", "05"]:
    path = f"{LANDING_PATH}yellow_tripdata_2023-{month}.parquet"
    df_nyc_taxi_data = spark.read.parquet(path)
    df_nyc_taxi_data = df_nyc_taxi_data.select(
        f.col("VendorID").cast(t.LongType()).alias("vendor_id"),
        f.col("passenger_count").cast(t.IntegerType()),
        f.col("total_amount").cast(t.DoubleType()),
        f.col("tpep_pickup_datetime").cast(t.TimestampType()),
        f.col("tpep_dropoff_datetime").cast(t.TimestampType()),
    )

    if df_nyc_taxi_data_final is None:
        df_nyc_taxi_data_final = df_nyc_taxi_data
    else:
        df_nyc_taxi_data_final = df_nyc_taxi_data_final.union(df_nyc_taxi_data)

df_nyc_taxi_data_final = df_nyc_taxi_data_final.filter(
    (f.col("tpep_pickup_datetime") >= "2023-01-01") &
    (f.col("tpep_pickup_datetime") <= "2023-05-31")
)

print(f"Total de linhas: {df_nyc_taxi_data_final.count():,}")
df_nyc_taxi_data_final.printSchema()

In [0]:
# Salva os arquivos como Delta Table no catálogo
df_nyc_taxi_data_final.write.format("delta").mode("overwrite").saveAsTable("workspace.nyc_taxi.yellow_trips")

In [0]:
%sql
-- Consultando a tabela para validacão
SELECT
  *
FROM
  workspace.nyc_taxi.yellow_trips
LIMIT 10

In [0]:
%sql
-- Adiciona descrição da tabela
ALTER TABLE
  workspace.nyc_taxi.yellow_trips
SET TBLPROPERTIES
  ('comment' = 'Tabela com dados de corridas de táxi em Nova York para o ano de 2023.');

-- Adiciona descrição das colunas
ALTER TABLE
  workspace.nyc_taxi.yellow_trips
ALTER COLUMN
  vendor_id
  COMMENT 'Código do fornecedor (1: Creative Mobile Technologies, 2: Curb Mobility, 6: Myle Technologies, 7: Helix).';

ALTER TABLE
  workspace.nyc_taxi.yellow_trips
ALTER COLUMN
  passenger_count
  COMMENT 'Número de passageiros no veículo.';

ALTER TABLE
  workspace.nyc_taxi.yellow_trips
ALTER COLUMN
  total_amount
  COMMENT 'Valor total cobrado do passageiro.';

ALTER TABLE
  workspace.nyc_taxi.yellow_trips
ALTER COLUMN
  tpep_pickup_datetime
  COMMENT 'Data e hora em que o taxímetro foi acionado.';

ALTER TABLE
  workspace.nyc_taxi.yellow_trips
ALTER COLUMN
  tpep_dropoff_datetime
  COMMENT 'Data e hora em que o taxímetro foi desligado.';